In [ ]:
import torch
torch.cuda.is_available()

In [ ]:
!nvidia-smi

In [ ]:
!pip install datasets torchvision torch scikit-learn matplotlib

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.optim as optim

from datasets import load_dataset

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
dataset = load_dataset(
    "dpdl-benchmark/plant_village",
    split="train"
)

print(dataset)

In [ ]:
print(dataset)

In [ ]:
sample = dataset[0]

print(sample)

In [ ]:
plt.imshow(sample["image"])
plt.title(sample["label"])
plt.axis("off")

label distribution visualization

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

labels = [sample["label"] for sample in dataset]

label_counts = Counter(labels)

plt.figure(figsize=(12,5))
plt.bar(label_counts.keys(), label_counts.values())

plt.xlabel("Disease Label")
plt.ylabel("Number of Images")
plt.title("Distribution of Plant Disease Labels")

plt.xticks(list(label_counts.keys()))

plt.show()

Splitting the dataset

In [ ]:
dataset_split = dataset.train_test_split(test_size=0.2, seed=42)

train_ds = dataset_split["train"]
test_ds = dataset_split["test"]

print(len(train_ds), len(test_ds))

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

train_labels = [sample["label"] for sample in train_ds]

counts = Counter(train_labels)

plt.figure(figsize=(12,5))
plt.bar(counts.keys(), counts.values())

plt.xlabel("Disease Label")
plt.ylabel("Number of Images")
plt.title("Training Data Label Distribution")

plt.show()

Image Transformations

In [ ]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

Creating a PyTorch Dataset Wrapper

In [ ]:


class PlantDiseaseDataset(torch.utils.data.Dataset):

    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):

        sample = self.dataset[idx]

        image = sample["image"]
        label = sample["label"]

        if self.transform:
            image = self.transform(image)

        return image, label

Creating Training and Testing Datasets

In [ ]:
train_dataset = PlantDiseaseDataset(train_ds, transform)
test_dataset = PlantDiseaseDataset(test_ds, transform)

Creating DataLoaders

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

Verifying the Pipeline

In [ ]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

loading ResNet18

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights="IMAGENET1K_V1")

In [ ]:
num_features = model.fc.in_features

model.fc = nn.Linear(num_features, 38)

In [ ]:
model = model.to(device)

Defining Loss Function

In [ ]:
criterion = nn.CrossEntropyLoss()

Defining Optimizer

In [ ]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.0003)

training loop

In [ ]:
epochs = 5

for epoch in range(epochs):

    model.train()
    running_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

Evaluation

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

confusion matrics

In [ ]:
from sklearn.metrics import classification_report

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds))

Saving the model

In [ ]:
torch.save(model.state_dict(), "plant_disease_resnet18.pth")

In [ ]:
from google.colab import output
output.clear()